Firstly, we install the rquired libraries.

In [1]:
!pip install transformers pandas torch

now, we create our incident dataset.

In [2]:
import pandas as pd

data = [
    "Accident on highway near junction 5, two lanes blocked, heavy congestion",
    "Road construction causing slow traffic in city center",
    "Traffic signal not working at main intersection",
    "Minor accident blocking one lane, moderate delay",
    "Heavy rain causing slow movement across all lanes"
]

df = pd.DataFrame(data, columns=["incident_text"])
df

,incident_text
0,"Accident on highway near junction 5, two lanes..."
1,Road construction causing slow traffic in city...
2,Traffic signal not working at main intersection
3,"Minor accident blocking one lane, moderate delay"
4,Heavy rain causing slow movement across all lanes


now that we have our dataset, we run the lare langauge model Flan-t5

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Now, we define the incident analysis function.

In [19]:
def analyze_incident(text):

    prompt = f"""
Classify the type of traffic incident.

Choose ONLY one:
accident, roadwork, signal_failure, weather, event, other

Incident: {text}
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=10)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()

    return response

Now, Let's build RULE-BASED EXTRACTION

In [21]:
def extract_features(text):
    text = text.lower()

    # incident type from LLM
    incident_type = analyze_incident(text)

    # severity
    if "heavy" in text or "severe" in text:
        severity = "high"
    elif "moderate" in text:
        severity = "medium"
    else:
        severity = "low"

    # lanes blocked
    if "two" in text or "2" in text:
        lanes = 2
    elif "one" in text or "1" in text:
        lanes = 1
    else:
        lanes = 0

    # impact
    if "congestion" in text:
        impact = "congestion"
    elif "slow" in text:
        impact = "delay"
    else:
        impact = "low impact"

    return {
        "incident_type": incident_type,
        "severity": severity,
        "lanes_blocked": lanes,
        "expected_impact": impact
    }

Let's apply to the dataset.

In [22]:
df["structured"] = df["incident_text"].apply(extract_features)

structured_df = pd.json_normalize(df["structured"])
final_df = pd.concat([df, structured_df], axis=1)

final_df.head()

,incident_text,structured,incident_type,severity,lanes_blocked,expected_impact
0,"Accident on highway near junction 5, two lanes...","{'incident_type': 'accident', 'severity': 'hig...",accident,high,2,congestion
1,Road construction causing slow traffic in city...,"{'incident_type': 'event', 'severity': 'low', ...",event,low,0,delay
2,Traffic signal not working at main intersection,"{'incident_type': 'accident', 'severity': 'low...",accident,low,0,low impact
3,"Minor accident blocking one lane, moderate delay","{'incident_type': 'event', 'severity': 'medium...",event,medium,1,low impact
4,Heavy rain causing slow movement across all lanes,"{'incident_type': 'event', 'severity': 'high',...",event,high,0,delay


Now, it's time to build the decision logic.

In [23]:
def decision_logic(row):

    if row["incident_type"] == "accident":
        if row["severity"] == "high":
            return "Reroute traffic + emergency response"
        else:
            return "Adjust signals"

    elif row["incident_type"] == "roadwork":
        return "Pre-planned diversion"

    elif row["incident_type"] == "signal_failure":
        return "Manual control"

    else:
        return "Monitor traffic"

In [24]:
final_df["decision"] = final_df.apply(decision_logic, axis=1)

In [25]:
final_df[["incident_text", "incident_type", "severity", "decision"]]

,incident_text,incident_type,severity,decision
0,"Accident on highway near junction 5, two lanes...",accident,high,Reroute traffic + emergency response
1,Road construction causing slow traffic in city...,event,low,Monitor traffic
2,Traffic signal not working at main intersection,accident,low,Adjust signals
3,"Minor accident blocking one lane, moderate delay",event,medium,Monitor traffic
4,Heavy rain causing slow movement across all lanes,event,high,Monitor traffic


finally, we add the confidence scores to the model.

In [26]:
def confidence_score(row):
    score = 0.5

    if row["incident_type"] in ["accident", "roadwork"]:
        score += 0.2

    if row["severity"] == "high":
        score += 0.2

    if row["lanes_blocked"] > 0:
        score += 0.1

    return round(min(score, 1.0), 2)

In [28]:
final_df["confidence"] = final_df.apply(confidence_score, axis=1)

Let's verify our system is working properly.

In [29]:
final_df[["incident_text", "incident_type", "severity", "lanes_blocked", "decision"]]

,incident_text,incident_type,severity,lanes_blocked,decision
0,"Accident on highway near junction 5, two lanes...",accident,high,2,Reroute traffic + emergency response
1,Road construction causing slow traffic in city...,event,low,0,Monitor traffic
2,Traffic signal not working at main intersection,accident,low,0,Adjust signals
3,"Minor accident blocking one lane, moderate delay",event,medium,1,Monitor traffic
4,Heavy rain causing slow movement across all lanes,event,high,0,Monitor traffic


In [30]:
test_cases = [
    "Massive accident blocking three lanes with heavy congestion",
    "Traffic signal failure causing delays",
    "Roadwork causing moderate traffic slowdown"
]

test_df = pd.DataFrame(test_cases, columns=["incident_text"])

test_df["structured"] = test_df["incident_text"].apply(extract_features)

test_structured = pd.json_normalize(test_df["structured"])
test_final = pd.concat([test_df, test_structured], axis=1)

test_final["decision"] = test_final.apply(decision_logic, axis=1)

test_final

,incident_text,structured,incident_type,severity,lanes_blocked,expected_impact,decision
0,Massive accident blocking three lanes with hea...,"{'incident_type': 'event', 'severity': 'high',...",event,high,0,congestion,Monitor traffic
1,Traffic signal failure causing delays,"{'incident_type': 'event', 'severity': 'low', ...",event,low,0,low impact,Monitor traffic
2,Roadwork causing moderate traffic slowdown,"{'incident_type': 'event', 'severity': 'medium...",event,medium,0,delay,Monitor traffic
